# 🔋 Battery RUL & Degradation Forecasting (Machine Learning)

This notebook provides a modular **Machine Learning Forecasting Pipeline** for the NASA Battery dataset.
You have full control over:
1. **Hyperparameters:** Directly edit the hyperparameter dictionaries for each model (XGBoost, Random Forest, SVR, etc.).
2. **Loss Function & Metrics:** Define custom loss functions (e.g. Asymmetric Penalties, Custom RMSE/MAE, Weighted Loss).
3. **Features:** Choose which operational signals to feed into the model.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nasa_battery import (
    BatteryDataLoader,
    BatteryMLPipeline,
    prepare_rul_dataset,
    calculate_metrics
)

sns.set_theme(style='whitegrid', palette='deep')
print('Imports successful! Ready to build ML models.')

## 1. Load Data & Prepare Features
- **Training Cells:** `B0005`, `B0006`, `B0007` (Room temp, 2 A discharge)
- **Testing Cell:** `B0018` (Out-of-sample cell under same conditions)

In [ ]:
data_dir = '../5. Battery Data Set'
loader = BatteryDataLoader(data_root=data_dir)

# You can change the EOL threshold (default 1.40 Ah = 30% fade from 2.0 Ah)
EOL_THRESHOLD = 1.40

train_batteries = ['B0005', 'B0006', 'B0007']
test_batteries = ['B0018']

train_df = prepare_rul_dataset(loader, train_batteries, eol_threshold=EOL_THRESHOLD)
test_df = prepare_rul_dataset(loader, test_batteries, eol_threshold=EOL_THRESHOLD)

print(f'Training dataset: {len(train_df)} cycles across {train_batteries}')
print(f'Testing dataset:  {len(test_df)} cycles for {test_batteries}')
train_df.head()

## 2. 🎛️ Define Your Custom Loss & Metric Function
> **YOUR INPUT HERE:** You can edit the function below to implement any loss or penalty function you want to evaluate.

In [ ]:
def my_custom_loss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Define your custom loss function here.
    
    Example below: Asymmetric penalty where predicting late (overestimating remaining battery life)
    is penalized 2x more heavily than predicting early (preventing unexpected battery failure).
    """
    errors = np.array(y_pred) - np.array(y_true)
    # If error > 0: predicted RUL is too high (late prediction penalty)
    # If error < 0: predicted RUL is too low (early prediction penalty)
    weights = np.where(errors > 0, 2.0, 1.0)
    custom_loss = np.mean(weights * (errors ** 2))
    return float(custom_loss)

# Test the custom loss on dummy values
print('Custom loss test:', my_custom_loss(np.array([50, 20]), np.array([55, 18])))

## 3. ⚙️ Define Your Model & Hyperparameters
> **YOUR INPUT HERE:** Select your model type (`'xgboost'`, `'random_forest'`, `'svr'`, `'gradient_boosting'`, `'ridge'`) and customize the hyperparameters dictionary.

In [ ]:
# Choose model type: 'xgboost', 'random_forest', 'svr', 'gradient_boosting', 'ridge'
SELECTED_MODEL = 'xgboost'

# --------------------------------------------------------------------------
# Define Hyperparameters for each model (Tweak these values!)
# --------------------------------------------------------------------------
HYPERPARAMETERS = {
    'xgboost': {
        'n_estimators': 150,      # Number of gradient boosted trees
        'max_depth': 4,           # Maximum tree depth
        'learning_rate': 0.05,    # Step size shrinkage
        'subsample': 0.8,         # Subsample ratio of training instances
        'colsample_bytree': 0.8,  # Subsample ratio of columns per tree
        'random_state': 42
    },
    'random_forest': {
        'n_estimators': 100,
        'max_depth': 6,
        'min_samples_split': 2,
        'random_state': 42
    },
    'svr': {
        'C': 10.0,                # Regularization parameter
        'epsilon': 0.1,           # Epsilon in the epsilon-SVR model
        'kernel': 'rbf',          # 'rbf', 'linear', or 'poly'
        'gamma': 'scale'
    },
    'gradient_boosting': {
        'n_estimators': 120,
        'learning_rate': 0.05,
        'max_depth': 3,
        'random_state': 42
    }
}

# Select input features and target
FEATURES = ['cycle_num', 'duration_sec', 'v_min', 'v_max', 'temp_max']
TARGET = 'RUL'  # or 'capacity' / 'SoH'

current_params = HYPERPARAMETERS.get(SELECTED_MODEL, {})
print(f'Selected Model: {SELECTED_MODEL}')
print(f'Hyperparameters: {current_params}')
print(f'Features: {FEATURES}')

## 4. Train Model & Run Evaluation

In [ ]:
# Initialize pipeline
pipeline = BatteryMLPipeline(
    features=FEATURES,
    target=TARGET,
    scale_features=True
)

# Fit model
pipeline.fit(train_df, model_type=SELECTED_MODEL, hyperparams=current_params)

# Evaluate on unseen test battery B0018
metrics, test_preds = pipeline.evaluate(test_df, custom_loss_fn=my_custom_loss)

print('='*50)
print(f'MODEL EVALUATION RESULTS ({SELECTED_MODEL.upper()} on B0018):')
print('='*50)
for metric_name, value in metrics.items():
    print(f'  {metric_name:<15}: {value:.4f}')
print('='*50)

## 5. Visualizing RUL Forecasts

In [ ]:
test_clean = test_df.dropna(subset=FEATURES + [TARGET]).copy()
test_clean['Predicted_RUL'] = test_preds

fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=120)

# 1. Actual vs Predicted RUL trajectory
axes[0].plot(test_clean['cycle_num'], test_clean['RUL'], label='Ground Truth RUL', color='black', lw=2.5)
axes[0].plot(test_clean['cycle_num'], test_clean['Predicted_RUL'], label=f'Predicted ({SELECTED_MODEL})', color='royalblue', lw=2, linestyle='--')
axes[0].set_title('RUL Forecasting on Unseen Battery B0018', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Discharge Cycle Number')
axes[0].set_ylabel('Remaining Useful Life (Cycles)')
axes[0].legend(frameon=True)

# 2. Prediction Error (Residuals)
error = test_preds - test_clean['RUL'].values
axes[1].plot(test_clean['cycle_num'], error, color='crimson', lw=1.5, label='Error (Pred - True)')
axes[1].axhline(0, color='gray', linestyle=':', lw=1.2)
axes[1].fill_between(test_clean['cycle_num'], 0, error, color='crimson', alpha=0.15)
axes[1].set_title('Forecast Residual Error across Aging Cycle', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Discharge Cycle Number')
axes[1].set_ylabel('Error (Cycles)')
axes[1].legend(frameon=True)

plt.tight_layout()
plt.show()

## 6. Multi-Model Benchmark Comparison Table
Compare all models side-by-side using your current hyperparameter settings and custom loss.

In [ ]:
models_to_compare = ['xgboost', 'random_forest', 'svr', 'gradient_boosting']
comparison_results = []

plt.figure(figsize=(10, 5), dpi=120)
plt.plot(test_clean['cycle_num'], test_clean['RUL'], label='Ground Truth RUL', color='black', lw=3)

for m in models_to_compare:
    pipe = BatteryMLPipeline(features=FEATURES, target=TARGET, scale_features=True)
    params = HYPERPARAMETERS.get(m, {})
    pipe.fit(train_df, model_type=m, hyperparams=params)
    m_metrics, m_preds = pipe.evaluate(test_df, custom_loss_fn=my_custom_loss)
    m_metrics['Model'] = m.upper()
    comparison_results.append(m_metrics)
    plt.plot(test_clean['cycle_num'], m_preds, label=f'{m.upper()}', lw=1.5, linestyle='--')

plt.title('Benchmark Comparison of ML Models on Battery B0018', fontsize=13, fontweight='bold')
plt.xlabel('Discharge Cycle Number')
plt.ylabel('RUL (Cycles)')
plt.legend()
plt.show()

# Display summary table
comp_df = pd.DataFrame(comparison_results).set_index('Model')
comp_df[['RMSE', 'MAE', 'R2', 'Custom_Loss', 'PHM_Score']]